# 🧠 Om-Think Fine-Tuning

Fine-tunes **Qwen2.5-Coder-3B-Instruct** with LoRA to create **Om-Think** — the reasoning engine for Om Agent.

**Requirements:**
- Google Colab with T4 GPU (free tier works)
- HuggingFace token (for pushing the model)
- ~20-30 minutes training time

**Steps:**
1. Set your HuggingFace token below
2. Runtime → Change runtime type → T4 GPU
3. Run All cells
4. Model auto-pushes to HuggingFace when done

In [ ]:
# ═══════════════════════════════════════════
# CONFIG — Set these before running
# ═══════════════════════════════════════════

HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # Get from: huggingface.co/settings/tokens
OUTPUT_HUB = "JohanKira/om-think-v1"  # Where to push the trained adapter

# Training hyperparameters (tested defaults — no need to change)
MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
MAX_SEQ = 1024
LORA_R = 16
LORA_ALPHA = 32
EPOCHS = 5
LR = 2e-4

In [ ]:
# ═══════════════════════════════════════════
# CELL 1: Install dependencies
# ═══════════════════════════════════════════

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub

print("\n✅ Dependencies installed")

In [ ]:
# ═══════════════════════════════════════════
# CELL 2: Download training data
# ═══════════════════════════════════════════

!wget -q https://raw.githubusercontent.com/KdarshUchiha/om-model/main/data/think/think_training_data.jsonl -O training_data.jsonl
!wget -q https://raw.githubusercontent.com/KdarshUchiha/om-model/main/data/think/domain_training_data.jsonl -O domain_data.jsonl

# Merge both datasets
!cat domain_data.jsonl >> training_data.jsonl

import json
with open('training_data.jsonl') as f:
    count = sum(1 for _ in f)
print(f"\n✅ Training data ready: {count} examples")

In [ ]:
# ═══════════════════════════════════════════
# CELL 3: Load model + apply LoRA
# ═══════════════════════════════════════════

from unsloth import FastLanguageModel
from huggingface_hub import login

login(token=HF_TOKEN)

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=MAX_SEQ, load_in_4bit=True
)

print("Applying LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print("\n✅ Model loaded with LoRA")

In [ ]:
# ═══════════════════════════════════════════
# CELL 4: Prepare dataset
# ═══════════════════════════════════════════

import json
from datasets import Dataset

raw = [json.loads(l) for l in open("training_data.jsonl") if l.strip()]
print(f"Loaded {len(raw)} examples")

def fmt(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)}

dataset = Dataset.from_list(raw).map(fmt, remove_columns=["messages"])
print(f"\n✅ Dataset formatted: {len(dataset)} examples")
print(f"Sample (first 200 chars): {dataset[0]['text'][:200]}...")

In [ ]:
# ═══════════════════════════════════════════
# CELL 5: Train
# ═══════════════════════════════════════════

from trl import SFTTrainer
from transformers import TrainingArguments

print("🚀 Starting training...")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning rate: {LR}")
print(f"   LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"   Max sequence length: {MAX_SEQ}")
print()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    packing=False,
    args=TrainingArguments(
        output_dir="./om-think-lora",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=LR,
        warmup_steps=5,
        optim="adamw_8bit",
        fp16=True,
        bf16=False,
        logging_steps=10,
        report_to="none",
    ),
)

trainer.train()
print("\n✅ Training complete!")

In [ ]:
# ═══════════════════════════════════════════
# CELL 6: Push to HuggingFace
# ═══════════════════════════════════════════

print(f"Pushing adapter to {OUTPUT_HUB}...")
model.push_to_hub(OUTPUT_HUB)
tokenizer.push_to_hub(OUTPUT_HUB)
print(f"\n✅ Model pushed to https://huggingface.co/{OUTPUT_HUB}")
print(f"\n🎯 Next step: Deploy serve-think/ directory as a HuggingFace Space")
print(f"   The Space will auto-load this adapter from {OUTPUT_HUB}")

In [ ]:
# ═══════════════════════════════════════════
# CELL 7 (Optional): Test the model
# ═══════════════════════════════════════════

import torch

FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "You are Om-Think — The Divine Reasoning Engine. Given a user's request, produce a comprehensive structured plan BEFORE any code is written. Think deeply about WHY, HOW, TRADEOFFS, EDGE CASES, and DECOMPOSITION."},
    {"role": "user", "content": "I want to build a real-time chat app. Walk me through your thinking."},
]

input_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=1024, temperature=0.7, do_sample=True)

response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("═" * 60)
print("OM-THINK TEST RESPONSE:")
print("═" * 60)
print(response[:3000])